In [ ]:
# Paths are configured via environment variables, see src/common/utils/src/utils/paths.py
import os
from utils.paths import DATA_ROOT, MATERIALS_DIR

# Experiment 1.1 compare with baseline methods

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm

font_size = 12
verdana_path = os.environ.get('VERDANA_TTF', 'Verdana.ttf')
verdana_font = fm.FontProperties(fname=verdana_path, size=font_size)
colors = ['#1F77B4', '#2CA02C', '#FF7F0E' , '#9467BD', '#D62728' ]

object_name = 'PLANT'
planner_name = ['random_global', 'predefined_global', 'ssl_nbv', 'voxel_global_nbv', 'ssl_semantic_nbv']
planner_plot_name = ['Random','Pre-Defined','SSL-NBV','Voxel-NBV','SSL-Semantic-NBV']

cov_index = ['cov_0','cov_1','cov_2', 'cov_3', 'cov_4', 'cov_5', 'cov_6', 'cov_7', 'cov_8', 'cov_9', 'cov_10', 'cov_11', 'cov_12', 'cov_13', 'cov_14']

plant_369_max_coverage_NODE = {
    'mean': 0.5996834709463044,
    'std': 0.054914641,
}
plant_369_max_coverage_FRUIT = {
    'mean': 0.507947028,
    'std': 0.045478395,
}
plant_369_max_coverage_PLANT = {
    'mean': 0.88271954,
    'std': 0.063818977,
}
number_runs_369 = 150

df = pd.read_csv(MATERIALS_DIR + '/prilinary_test/compare_diff_planners/%s/results.csv' % object_name)

auc_list = []
auc_low_list = []
auc_high_list = []
plt.figure(figsize=(5.5, 3.5))
for idx, planner in enumerate(planner_name):
    df_planner = df[df['planner']==planner]
    coverage = df_planner[cov_index]
    plt.plot(coverage.mean(axis=0), label=planner_plot_name[idx], color=colors[idx])
    ci = 1.96 * (coverage.std(axis=0) / np.sqrt(len(coverage)))
    plt.fill_between(range(len(coverage.mean(axis=0))), 
                     coverage.mean(axis=0) - ci, 
                     coverage.mean(axis=0) + ci, 
                     alpha=0.2, color=colors[idx])
    
    AUC_average = df_planner['auc'].mean(axis=0)
    AUC_std = df_planner['auc'].std(axis=0)
    # Calculate 95% confidence interval instead of using std directly
    AUC_low = AUC_average - 1.96 * (AUC_std / np.sqrt(len(df_planner)))
    AUC_high = AUC_average + 1.96 * (AUC_std / np.sqrt(len(df_planner)))
    auc_list.append(AUC_average)
    auc_low_list.append(AUC_low)
    auc_high_list.append(AUC_high)
    if planner == 'ssl_nbv':
        print(df_planner['auc'])
        
    print(f"{planner_plot_name[idx]}: AUC={AUC_average:.4f}, Low={AUC_low:.4f}, High={AUC_high:.4f}")

#get object basd on the object_name
MAX_COVERAGE = eval('plant_369_max_coverage_%s' % object_name)
max_coverage_mean = MAX_COVERAGE['mean']
max_coverage_std = MAX_COVERAGE['std']

#copy the max_coverage_mean multiple times as the same length as the coverage
max_coverage_mean_list = [max_coverage_mean]*len(coverage.mean(axis=0))
max_coverage_low_list = [max_coverage_mean - (1.96 * max_coverage_std/np.sqrt(number_runs_369))]*len(coverage.mean(axis=0))
max_coverage_high_list = [max_coverage_mean + (1.96 * max_coverage_std/np.sqrt(number_runs_369))]*len(coverage.mean(axis=0))

x_ticks = [str(i+1) for i in range(len(coverage.mean(axis=0)))]

plt.plot(max_coverage_mean_list, color='black', linestyle='--')
plt.fill_between(range(len(max_coverage_mean_list)), 
                 max_coverage_low_list, 
                 max_coverage_high_list, 
                 alpha=0.2)

# fill an region as the 95% confidence interval
plt.xticks(range(len(coverage.mean(axis=0))), x_ticks, fontproperties=verdana_font)
plt.ylim(0, 0.95) # PLANT: 0.95, FRUIT: 0.6, NODE: 0.7
plt.yticks(np.arange(0, 0.96, 0.1), fontproperties=verdana_font)
plt.xlabel('Number of views [-]', fontproperties=verdana_font)
plt.ylabel('Coverage [%]', fontproperties=verdana_font)
plt.grid()
plt.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig(MATERIALS_DIR + f'/prilinary_test/compare_diff_planners/coverage_{object_name}.png')
plt.show()

# plot the AUC for different planners using vertical bar plot
plt.figure(figsize=(5.5, 4))
#set colors align as the above plot

plt.bar(planner_plot_name, auc_list, color=colors, yerr=[np.array(auc_list) - np.array(auc_low_list), np.array(auc_high_list) - np.array(auc_list)], capsize=5)
plt.xlabel('Method [-]', fontproperties=verdana_font, fontweight='bold')
plt.ylabel('AUC@VR [-]', fontproperties=verdana_font, fontweight='bold')
plt.xticks(fontproperties=verdana_font, rotation=20)
plt.yticks(fontproperties=verdana_font)
plt.ylim(0, 0.7) # PLANT: 0.7, FRUIT: 0.5, NODE: 0.5
#grid only on the y axis
plt.grid(axis='y')
plt.tight_layout()
plt.savefig(MATERIALS_DIR + f'/prilinary_test/compare_diff_planners/AUC_{object_name}.png')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm

font_size = 12
verdana_path = os.environ.get('VERDANA_TTF', 'Verdana.ttf')
verdana_font = fm.FontProperties(fname=verdana_path, size=font_size)
colors = ['#1F77B4', '#2CA02C', '#FF7F0E' , '#9467BD', '#D62728' ]

object_name = ['PLANT', 'FRUIT', 'NODE']
planner_name = ['random_global', 'predefined_global', 'ssl_nbv', 'voxel_global_nbv', 'ssl_semantic_nbv']
planner_plot_name = ['Random','Pre-Defined','SSL-NBV','Voxel-NBV','SSL-Semantic-NBV']

cov_index = ['cov_0', 'cov_4', 'cov_9', 'cov_14']

plant_369_max_coverage_NODE = {
    'mean': 0.5996834709463044,
    'std': 0.054914641,
}
plant_369_max_coverage_FRUIT = {
    'mean': 0.507947028,
    'std': 0.045478395,
}
plant_369_max_coverage_PLANT = {
    'mean': 0.88271954,
    'std': 0.063818977,
}
number_runs_369 = 150

for object in object_name:
    df = pd.read_csv(MATERIALS_DIR + '/prilinary_test/compare_diff_planners/%s/results.csv' % object)

    for idx, planner in enumerate(planner_name):
        df_planner = df[df['planner']==planner]
        coverage = df_planner[cov_index]    
        AUC_average = df_planner['auc'].mean(axis=0)

        #get object basd on the object_name
        MAX_COVERAGE = eval('plant_369_max_coverage_%s' % object)
        max_coverage_mean = MAX_COVERAGE['mean']

        print(f"{object}: {planner_plot_name[idx]}: Coverage={np.array(coverage.mean(axis=0))}, AUC={np.array(AUC_average):.4f}, Max Coverage={np.array(max_coverage_mean):.4f}")


# Experiment 1.2 Evaluate target-semantic mechanism

In [ ]:
#draw me a confusion matrix using fake results

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


train_targets = ['PLANT', 'FRUIT', 'NODE']
recon_targets = ['PLANT', 'FRUIT', 'NODE']

dir = MATERIALS_DIR + '/prilinary_test/test_semantic_aware'

results = dict()
for train_t in train_targets:
    for recon_t in recon_targets:
        csv_file = os.path.join(dir, f'train_{train_t}_recon_{recon_t}.csv')
        df = pd.read_csv(csv_file)
        df_auc = df['auc'].mean(axis=0)
        print(f'{train_t} {recon_t} {df_auc}')

        results[f'{train_t}_{recon_t}'] = df_auc

# Define class labels for semantic segmentation
class_labels = ['PLANT', 'NODE', 'FRUIT']

# Create a confusion matrix
results_re_ordered = np.zeros((len(class_labels), len(class_labels)))
for train_t in class_labels:
    for recon_t in class_labels:
        results_re_ordered[class_labels.index(train_t), class_labels.index(recon_t)] = results[f'{train_t}_{recon_t}']

plt.figure(figsize=(6, 5))
ax = sns.heatmap(results_re_ordered, annot=True, fmt='.2f', cmap='Reds', 
                 xticklabels=class_labels, yticklabels=class_labels, cbar=False,
                 annot_kws={'size': 16})

# Move x-axis labels to top
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')

# Increase font size of tick labels
ax.tick_params(axis='x', labelsize=12)
ax.tick_params(axis='y', labelsize=12)

plt.xlabel('Train on #', fontsize=16)
plt.ylabel('Test on #', fontsize=16)

plt.tight_layout()
plt.savefig(MATERIALS_DIR + f'/prilinary_test/test_semantic_aware/confusion_matrix.png')
plt.show()

# Experiment 1.3 Abalation test

In [ ]:
# draw the coverage of the proposed method and the baseline methods
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np  
import matplotlib.font_manager as fm

font_size = 12
verdana_path = os.environ.get('VERDANA_TTF', 'Verdana.ttf')
verdana_font = fm.FontProperties(fname=verdana_path, size=font_size)
#colors = ['#1F77B4', '#2CA02C', '#FF7F0E' , '#9467BD', '#D62728']
colors = ['#1F77B4', '#2CA02C', '#FF7F0E' , '#D62728']


results_dir = MATERIALS_DIR + '/prilinary_test/ablation_test'
targets = ['PLANT', 'FRUIT', 'NODE']

#methods = ['ssl_semantic_nbv_NOLF, 'ssl_semantic_nbv_NOPE', 'ssl_semantic_nbv_NOPE_NOLF', 'ssl_semantic_nbv_NOPE_NOLF_NOIG', 'ssl_semantic_nbv']
#plot_lables = ['No $\mathcal{L}$','No VTN','No VTN+$\mathcal{L}$','No VTN+$\mathcal{L}$+$\mathcal{f}^{IG}$',   'SSL-Semantic-NBV']


# methods = ['ssl_semantic_nbv_NOLF', 'ssl_semantic_nbv_NOPE_NOLF', 'ssl_semantic_nbv_NOPE_NOLF_NOIG', 'ssl_semantic_nbv']
# plot_lables = ['No $\mathcal{L}$','No VTN+$\mathcal{L}$','No VTN+$\mathcal{L}$+$\mathcal{f}^{IG}$',   'SSL-Semantic-NBV']

methods = ['ssl_semantic_nbv_NOPE', 'ssl_semantic_nbv_NOPE_NOLF', 'ssl_semantic_nbv_NOPE_NOLF_NOIG', 'ssl_semantic_nbv']
plot_lables = ['No VTN','No VTN+$\mathcal{L}$','No VTN+$\mathcal{L}$+$\mathcal{f}^{IG}$',   'SSL-Semantic-NBV']

# how to type script font L in python plot
# use font_manager to load the font

ckpts = np.arange(0, 2001, 200)

for target in targets:
    plt.figure(figsize=(5.5, 4))
    for method in methods:
        results_file = os.path.join(results_dir, f'{method}_{target}.csv')
        results = pd.read_csv(results_file)

        auc_mean_list = []
        auc_low_list = []
        auc_high_list = []
        for ckpt in ckpts:
            ckpt_table = results[results['ckpt'] == ckpt]
            auc_mean_list.append(ckpt_table['auc'].mean(axis=0))
            auc_low_list.append(ckpt_table['auc'].mean(axis=0) - 1.96 * (ckpt_table['auc'].std(axis=0) / np.sqrt(len(ckpt_table))))
            auc_high_list.append(ckpt_table['auc'].mean(axis=0) + 1.96 * (ckpt_table['auc'].std(axis=0) / np.sqrt(len(ckpt_table))))
            
        plt.plot(ckpts, auc_mean_list, marker='o', markersize=5, label=plot_lables[methods.index(method)], color=colors[methods.index(method)])
        # add 95% confidence interval
        plt.fill_between(ckpts, auc_low_list, auc_high_list, alpha=0.2)

    plt.xlabel('Number training cycles [-]', fontproperties=verdana_font)
    plt.xticks(ckpts, fontproperties=verdana_font, fontsize=10)
    plt.yticks(fontproperties=verdana_font, fontsize=10)
    plt.ylim(0, 0.65)
    plt.ylabel('AUC@VR [-]', fontproperties=verdana_font)
    #set x_ticks rotation to 20
    # plt.xticks(rotation=20)
    plt.grid(axis='y')
    plt.legend(loc='lower right', fontsize=10)
    plt.tight_layout()
    plt.savefig(MATERIALS_DIR + f'/prilinary_test/ablation_test/ablation_{target}.png')
    plt.show()


# Experiment 1.4 Evaluate occlusion handling

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np  
import matplotlib.font_manager as fm

font_size = 12
verdana_path = os.environ.get('VERDANA_TTF', 'Verdana.ttf')
verdana_font = fm.FontProperties(fname=verdana_path, size=font_size)
colors = ['#1F77B4', '#2CA02C', '#FF7F0E' , '#9467BD', '#D62728']

results_normal = MATERIALS_DIR + '/prilinary_test/compare_diff_planners'
results_occlusion = MATERIALS_DIR + '/prilinary_test/compare_diff_planners_heavy_occlusion'

targets = ['PLANT', 'FRUIT', 'NODE']
planner_name = ['random_global', 'predefined_global', 'ssl_nbv', 'voxel_global_nbv', 'ssl_semantic_nbv']
planner_plot_name = ['Random','Pre-Defined','SSL-NBV','Voxel-NBV','SSL-Semantic-NBV']

for target in targets:
    print('======================target: %s=========================' % target)
    for method in planner_name:
        normal_df = pd.read_csv(os.path.join(results_normal, target, 'results.csv'))
        occlusion_df = pd.read_csv(os.path.join(results_occlusion, target, 'results.csv'))
        normal_auc = normal_df[normal_df['planner']==method]['auc'].mean()
        occlusion_auc = occlusion_df[occlusion_df['planner']==method]['auc'].mean()
        print(f'{method} normal auc: {normal_auc}, occlusion auc: {occlusion_auc}, relative decrease: {100 * (normal_auc - occlusion_auc) / normal_auc}%')





# Experiment #4 Evaluate pre-training on synthetic plants

In [ ]:
# draw the coverage of the proposed method and the baseline methods
import matplotlib.pyplot as plt
import numpy as np  
import os
import time

number_training = 2000
rounds =  np.arange(1, number_training+1)
methods = ['With digital plants pre-training', 'Without pre-training']

initial_auc_with_pre_training = 0.5
initial_auc_without_pre_training = 0.2

final_auc_with_pre_training = 0.95
final_auc_without_pre_training = 0.9

rate = 0.01

# Correct formula: initial + (final - initial) * (1 - exp(-rate * (x - 1)))
auc_with_pre_training = initial_auc_with_pre_training + (final_auc_with_pre_training - initial_auc_with_pre_training) * (1 - np.exp(-rate * (rounds - 1)))
auc_without_pre_training = initial_auc_without_pre_training + (final_auc_without_pre_training - initial_auc_without_pre_training) * (1 - np.exp(-rate/5 * (rounds - 1)))

# add noise to the auc_with_pre_training and auc_without_pre_training
auc_with_pre_training = auc_with_pre_training + np.random.normal(0, 0.03, size=len(rounds))
auc_without_pre_training = auc_without_pre_training + np.random.normal(0, 0.03, size=len(rounds))

plt.figure(figsize=(7, 5))
plt.plot(rounds, auc_with_pre_training, label='Digital plants pre-training')
plt.plot(rounds, auc_without_pre_training, label='Without pre-training')
plt.xlabel('Number of training rounds [-]', fontsize=14)
plt.ylabel('AUC [-]', fontsize=14)
plt.grid()
plt.legend()

# save the figure
plt.tight_layout()
plt.savefig(MATERIALS_DIR + f'/auc.png')
plt.show()


In [ ]:
# train_plants = [11, 12, 14, 15, 17, 18, 20]  ##select the plants for training
# valid_plants = [13, 16, 19]  
# all_plants = train_plants + valid_plants

# nbv_data_path = DATA_ROOT + '/data_smaller_fov_distance'

# x_range = np.arange(-20, 20 + 1, 10)
# y_range = np.arange(-20, 20 + 1, 10)
# z_range = np.arange(0, 20 + 1, 20)
# rotation_range = np.arange(0, 360, 90)

# h_range = np.arange(0, 9, 1)
# w_range = np.arange(0, 9, 1)

# def get_pcd(plant, rotation, position: list, hw: list):
#     pcd_path = os.path.join(
#         nbv_data_path,
#         "plant%s" % str(plant),
#         "xyzr_%s_%s_%s_%s" % (str(position[0]), str(position[1]), str(position[2]), str(rotation)),
#         'pcd',
#         "pcd_%s_%s.npy" % (str(hw[0]), str(hw[1])),
#     )
#     pcd = np.load(pcd_path)
#     return pcd

# for plant in all_plants:    
#     for x in x_range:
#         for y in y_range:
#             for z in z_range:
#                 for rotation in rotation_range:
#                     for h in h_range:
#                         for w in w_range:
#                             print(plant, rotation, [x, y, z], [h, w])
#                             pcd = get_pcd(plant, rotation, [x, y, z], [h, w])
#                             print(pcd.shape)

# Real world experiment compared different planners

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm

font_size = 12
verdana_path = os.environ.get('VERDANA_TTF', 'Verdana.ttf')
verdana_font = fm.FontProperties(fname=verdana_path, size=font_size)
colors = ['#1F77B4', '#2CA02C', '#FF7F0E' , '#9467BD', '#D62728' ]

object_name = 'NODE'
planner_name = ['random_global', 'predefined_global', 'ssl_nbv', 'voxel_global_nbv', 'ssl_semantic_nbv']
planner_plot_name = ['Random','Pre-Defined','SSL-NBV','Voxel-NBV','SSL-Semantic-NBV']

cov_index = ['cov_0','cov_1','cov_2', 'cov_3', 'cov_4', 'cov_5', 'cov_6', 'cov_7', 'cov_8', 'cov_9', 'cov_10', 'cov_11', 'cov_12', 'cov_13', 'cov_14']

df = pd.read_csv(MATERIALS_DIR + '/prilinary_test/real_world_experiments/%s/results.csv'%str(object_name))

auc_list = []
auc_low_list = []
auc_high_list = []
# plt.figure(figsize=(5.5, 3.5))
for idx, planner in enumerate(planner_name):
    df_planner = df[df['planner']==planner]
    coverage = df_planner[cov_index]
    
    AUC_average = df_planner['auc'].mean(axis=0)
    AUC_std = df_planner['auc'].std(axis=0)
    # Calculate 95% confidence interval instead of using std directly
    AUC_low = AUC_average - 1.96 * (AUC_std / np.sqrt(len(df_planner)))
    AUC_high = AUC_average + 1.96 * (AUC_std / np.sqrt(len(df_planner)))
    auc_list.append(AUC_average)
    auc_low_list.append(AUC_low)
    auc_high_list.append(AUC_high)
    
    print(f"{object_name}, {planner_plot_name[idx]}: AUC={AUC_average:.4f}, Low={AUC_low:.4f}, High={AUC_high:.4f}")

# plot the AUC for different planners using vertical bar plot
plt.figure(figsize=(5.5, 4))
plt.bar(planner_plot_name, auc_list, color=colors, yerr=[np.array(auc_list) - np.array(auc_low_list), np.array(auc_high_list) - np.array(auc_list)], capsize=5)
plt.xlabel('Method [-]', fontproperties=verdana_font, fontweight='bold')
plt.ylabel('AUC@VR [-]', fontproperties=verdana_font, fontweight='bold')
plt.xticks(fontproperties=verdana_font, rotation=20)
plt.yticks(fontproperties=verdana_font)
plt.ylim(0, 0.8) # PLANT: 0.7, FRUIT: 0.5, NODE: 0.5
plt.grid(axis='y')
plt.tight_layout()
plt.savefig(MATERIALS_DIR + f'/prilinary_test/real_world_experiments/REAL_AUC_{object_name}.png')
plt.show()

# Analysis inference speed

In [ ]:
# read the csv file and calculate the mean and std for the step_time column across targets.
# combine PLANT and FRUIT results, then compute mean/std per planner and relative decrease vs random.
import os
import pandas as pd

targets = ['PLANT', 'FRUIT']
planner_name = ['random_global', 'predefined_global', 'ssl_nbv', 'voxel_global_nbv', 'ssl_semantic_nbv']
planner_plot_name = ['Random','Pre-Defined','SSL-NBV','Voxel-NBV','SSL-Semantic-NBV']

dfs = []
for target in targets:
    df_target = pd.read_csv(MATERIALS_DIR + '/prilinary_test/compare_diff_planners/%s/results.csv' % target)
    df_target['target'] = target
    dfs.append(df_target)

df = pd.concat(dfs, ignore_index=True)
print('======================combined targets: %s=========================' % ','.join(targets))
random_step_time = df[df['planner']=='random_global']['step_time'].mean()

for method in planner_name:
    step_time = df[df['planner']==method]['step_time'].mean()
    step_time_std = df[df['planner']==method]['step_time'].std()
    if method == 'random_global' or random_step_time == 0 or pd.isna(random_step_time):
        relative_decrease = float('nan')
    else:
        relative_decrease = (random_step_time - step_time) / random_step_time * 100.0
    print(f'{method} step_time: {step_time:.4f} ± {step_time_std:.4f} seconds, relative decrease: {relative_decrease:.2f}%')

# Below reanalysis the results for the ssl-semantic-nbv to remove its color information

## compare with different planners

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm
import os
from scipy.stats import ttest_ind

font_size = 12
verdana_path = os.environ.get('VERDANA_TTF', 'Verdana.ttf')
verdana_font = fm.FontProperties(fname=verdana_path, size=font_size)
colors = ['#1F77B4', '#2CA02C', '#FF7F0E' , '#9467BD', '#D62728' ]

targets = ['PLANT', 'NODE', 'FRUIT']
experiment = 0
planner_name = ['random_global', 'predefined_global', 'ssl_nbv','voxel_global_nbv' ,'ssl_semantic_nbv_NOCOLOR']
planner_plot_name = ['Random','Pre-Defined','SSL-NBV','Voxel-NBV','SSL-Semantic-NBV']

cov_index = ['cov_0','cov_1','cov_2', 'cov_3', 'cov_4', 'cov_5', 'cov_6', 'cov_7', 'cov_8', 'cov_9', 'cov_10', 'cov_11', 'cov_12', 'cov_13', 'cov_14']

plant_369_max_coverage_NODE = {
    'mean': 0.5996834709463044,
    'std': 0.054914641,
}
plant_369_max_coverage_FRUIT = {
    'mean': 0.507947028,
    'std': 0.045478395,
}
plant_369_max_coverage_PLANT = {
    'mean': 0.88271954,
    'std': 0.063818977,
}
number_runs_369 = 150

results_root = MATERIALS_DIR + '/NO_COLOR_RESULTS'

for object_name in targets:
    results_dir = os.path.join(results_root, f'{experiment}_{object_name}')
    df = pd.read_csv(os.path.join(results_dir, 'results.csv'))

    auc_list = []
    auc_low_list = []
    auc_high_list = []
    plt.figure(figsize=(5.5, 3.5))
    ssl_semantic_nbv_cov = None
    ssl_semantic_nbv_idx = planner_name.index('ssl_semantic_nbv_NOCOLOR')
    all_coverages = []

    for idx, planner in enumerate(planner_name):
        df_planner = df[df['planner']==planner]
        coverage = df_planner[cov_index]
        all_coverages.append(coverage)
        plt.plot(coverage.mean(axis=0), label=planner_plot_name[idx], color=colors[idx])
        ci = 1.96 * (coverage.std(axis=0) / np.sqrt(len(coverage)))
        plt.fill_between(range(len(coverage.mean(axis=0))), 
                         coverage.mean(axis=0) - ci, 
                         coverage.mean(axis=0) + ci, 
                         alpha=0.2, color=colors[idx])
        
        AUC_average = df_planner['auc'].mean(axis=0)
        AUC_std = df_planner['auc'].std(axis=0)
        # Calculate 95% confidence interval instead of using std directly
        AUC_low = AUC_average - 1.96 * (AUC_std / np.sqrt(len(df_planner)))
        AUC_high = AUC_average + 1.96 * (AUC_std / np.sqrt(len(df_planner)))
        auc_list.append(AUC_average)
        auc_low_list.append(AUC_low)
        auc_high_list.append(AUC_high)
        print(f"{object_name} | {planner_plot_name[idx]}: AUC={AUC_average:.4f}, Low={AUC_low:.4f}, High={AUC_high:.4f}")

    # Add cross sign for ssl_semantic_nbv_NOCOLOR if significantly higher at each view
    ssl_semantic_nbv_cov = all_coverages[ssl_semantic_nbv_idx]
    x_vals = np.arange(len(cov_index))
    y_vals = ssl_semantic_nbv_cov.mean(axis=0).values
    for v in x_vals:
        p_values = []
        for idx, coverage in enumerate(all_coverages):
            if idx == ssl_semantic_nbv_idx:
                continue
            # T-test for this view
            t_stat, p_val = ttest_ind(ssl_semantic_nbv_cov[cov_index[v]], coverage[cov_index[v]], equal_var=False)
            p_values.append(p_val)
        # If all p-values < 0.05 and mean is higher than all others, add 'x'
        if all(p < 0.05 for p in p_values) and all(y_vals[v] > coverage[cov_index[v]].mean() for idx, coverage in enumerate(all_coverages) if idx != ssl_semantic_nbv_idx):
            plt.scatter(v, y_vals[v], marker='x', color='black', s=30, zorder=10)  # smaller font size for 'x'

    MAX_COVERAGE = eval('plant_369_max_coverage_%s' % object_name)
    max_coverage_mean = MAX_COVERAGE['mean']
    max_coverage_std = MAX_COVERAGE['std']

    max_coverage_mean_list = [max_coverage_mean]*len(coverage.mean(axis=0))
    max_coverage_low_list = [max_coverage_mean - (1.96 * max_coverage_std/np.sqrt(number_runs_369))]*len(coverage.mean(axis=0))
    max_coverage_high_list = [max_coverage_mean + (1.96 * max_coverage_std/np.sqrt(number_runs_369))]*len(coverage.mean(axis=0))

    x_ticks = [str(i+1) for i in range(len(coverage.mean(axis=0)))]

    plt.plot(max_coverage_mean_list, color='black', linestyle='--')
    plt.fill_between(range(len(max_coverage_mean_list)), 
                     max_coverage_low_list, 
                     max_coverage_high_list, 
                     alpha=0.2)

    if object_name == 'NODE':
        coverage_ymax = 0.7
    elif object_name == 'FRUIT':
        coverage_ymax = 0.6
    else:
        coverage_ymax = 0.95

    plt.xticks(range(len(coverage.mean(axis=0))), x_ticks, fontproperties=verdana_font)
    plt.ylim(0, coverage_ymax)
    plt.yticks(np.arange(0, coverage_ymax + 0.01, 0.1), fontproperties=verdana_font)
    plt.xlabel('Number of views [-]', fontproperties=verdana_font)
    plt.ylabel('Reconstruction Ratio [-]', fontproperties=verdana_font)
    plt.grid()
    plt.legend(loc='lower right', fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f'coverage_{object_name}.png'))
    plt.show()

    # plot the AUC for different planners using vertical bar plot
    plt.figure(figsize=(5.5, 4))
    plt.bar(planner_plot_name, auc_list, color=colors, yerr=[np.array(auc_list) - np.array(auc_low_list), np.array(auc_high_list) - np.array(auc_list)], capsize=5)
    plt.xlabel('Method [-]', fontproperties=verdana_font, fontweight='bold')
    plt.ylabel('AUC@VR [-]', fontproperties=verdana_font, fontweight='bold')
    plt.xticks(fontproperties=verdana_font, rotation=20)
    plt.yticks(fontproperties=verdana_font)
    plt.ylim(0, coverage_ymax)
    plt.grid(axis='y')
    # Add 'x' above bar if ssl_semantic_nbv_NOCOLOR is significantly higher than all others
    ssl_semantic_auc = df[df['planner']=='ssl_semantic_nbv_NOCOLOR']['auc']
    p_values_auc = []
    for j, other_planner in enumerate(planner_name):
        if other_planner == 'ssl_semantic_nbv_NOCOLOR':
            continue
        other_auc = df[df['planner']==other_planner]['auc']
        t_stat, p_val = ttest_ind(ssl_semantic_auc, other_auc, equal_var=False)
        p_values_auc.append(p_val)
    if all(p < 0.05 for p in p_values_auc) and all(ssl_semantic_auc.mean() > df[df['planner']==other_planner]['auc'].mean() for other_planner in planner_name if other_planner != 'ssl_semantic_nbv_NOCOLOR'):
        idx = ssl_semantic_nbv_idx
        plt.text(idx, auc_list[idx]+0.01, 'x', ha='center', va='bottom', color='black', fontsize=8, fontweight='bold')  # smaller font size for 'x'
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f'AUC_{object_name}_with_cross.png'))
    plt.show()


# Compare results on experiment 0 and 1

In [ ]:
# compare planners: print AUC per method for experiment 0 and 1 (no plots)
import os
import csv
import pandas as pd
import numpy as np

results_root = MATERIALS_DIR + '/NO_COLOR_RESULTS'
experiments = [0, 1]
targets = ['PLANT', 'FRUIT', 'NODE']
planner_name = ['random_global', 'predefined_global', 'ssl_nbv', 'voxel_global_nbv', 'ssl_semantic_nbv_NOCOLOR']
planner_plot_name = ['Random','Pre-Defined','SSL-NBV','Voxel-NBV','SSL-Semantic-NBV']

def read_results_csv(path):
    if not os.path.exists(path):
        return pd.DataFrame()
    with open(path, 'r', newline='') as f:
        reader = csv.reader(f)
        rows = list(reader)
    if not rows:
        return pd.DataFrame()
    header = rows[0]
    valid_rows = [r for r in rows[1:] if len(r) == len(header)]
    return pd.DataFrame(valid_rows, columns=header)

for experiment in experiments:
    print(f'======================experiment: {experiment}=========================')
    per_method_target_means = {method: [] for method in planner_name}
    for target in targets:
        results_dir = os.path.join(results_root, f'{experiment}_{target}')
        csv_path = os.path.join(results_dir, 'results.csv')
        df = read_results_csv(csv_path)
        print(f'-- target: {target} --')
        if df.empty:
            print('No data found')
            print()
            continue
        for method, method_name in zip(planner_name, planner_plot_name):
            auc_series = pd.to_numeric(df[df['planner']==method]['auc'], errors='coerce')
            auc_mean = auc_series.mean()
            auc_std = auc_series.std()
            if not np.isnan(auc_mean):
                per_method_target_means[method].append(auc_mean)
            print(f'{method_name} AUC: {auc_mean:.4f} ± {auc_std:.4f}')
        print()
    print('Average AUC across targets:')
    for method, method_name in zip(planner_name, planner_plot_name):
        target_means = per_method_target_means[method]
        if target_means:
            avg_across_targets = float(np.mean(target_means))
            std_across_targets = float(np.std(target_means, ddof=1)) if len(target_means) > 1 else 0.0
            print(f'{method_name} avg AUC: {avg_across_targets:.4f} ± {std_across_targets:.4f}')
        else:
            print(f'{method_name} avg AUC: n/a')
    print()

## inference speed

In [ ]:
# read the csv file and calculate the mean and std for the step_time column.
# analyze inference speed for experiment 0 using NO_COLOR_RESULTS.
import os
import pandas as pd

experiment = 0
targets = ['PLANT', 'NODE', 'FRUIT']
planner_name = ['random_global', 'predefined_global', 'ssl_nbv', 'voxel_global_nbv', 'ssl_semantic_nbv_NOCOLOR']
planner_plot_name = ['Random','Pre-Defined','SSL-NBV','Voxel-NBV','SSL-Semantic-NBV']

results_root = MATERIALS_DIR + '/NO_COLOR_RESULTS'

def print_stats(df, title):
    print(title)
    for method, method_name in zip(planner_name, planner_plot_name):
        step_time = df[df['planner']==method]['step_time'].mean()
        step_time_std = df[df['planner']==method]['step_time'].std()
        print(f'{method_name} step_time: {step_time:.4f} ± {step_time_std:.4f} seconds')
    print()

dfs = []
for target in targets:
    results_dir = os.path.join(results_root, f'{experiment}_{target}')
    df_target = pd.read_csv(os.path.join(results_dir, 'results.csv'))
    df_target['target'] = target
    dfs.append(df_target)
    print_stats(df_target, f'======================target: {target} (experiment 0)=========================')

df = pd.concat(dfs, ignore_index=True)
print_stats(df, '======================combined targets (experiment 0)=========================')

# Ablation test

In [ ]:
# draw the coverage of the proposed method and the baseline methods
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np  
import matplotlib.font_manager as fm

font_size = 12
verdana_path = os.environ.get('VERDANA_TTF', 'Verdana.ttf')
verdana_font = fm.FontProperties(fname=verdana_path, size=font_size)
#colors = ['#1F77B4', '#2CA02C', '#FF7F0E' , '#9467BD', '#D62728']
colors = ['#1F77B4', '#2CA02C', '#FF7F0E' , '#D62728']

results_root = MATERIALS_DIR + '/NO_COLOR_RESULTS/ablation_test'
experiment = 0
targets = ['PLANT', 'FRUIT', 'NODE']

methods = ['ssl_semantic_nbv_NOPE_NOCOLOR', 'ssl_semantic_nbv_NOPE_NOLF_NOCOLOR', 'ssl_semantic_nbv_NOPE_NOLF_NOIG_NOCOLOR', 'ssl_semantic_nbv_NOCOLOR']
plot_lables = ['No VTN','No VTN+$\\mathcal{L}$','No VTN+$\\mathcal{L}$+$\\mathcal{f}^{IG}$',   'SSL-Semantic-NBV']

# how to type script font L in python plot
# use font_manager to load the font

ckpts = np.arange(0, 2001, 200)

for target in targets:
    plt.figure(figsize=(5.5, 4))
    results_dir = os.path.join(results_root, f'{experiment}_{target}')
    results_file = os.path.join(results_dir, 'ablation_results.csv')
    results = pd.read_csv(results_file)

    for method in methods:
        method_table = results[results['planner'] == method]

        auc_mean_list = []
        auc_low_list = []
        auc_high_list = []
        for ckpt in ckpts:
            ckpt_table = method_table[method_table['ckpt'] == ckpt]
            auc_mean_list.append(ckpt_table['auc'].mean(axis=0))
            auc_low_list.append(ckpt_table['auc'].mean(axis=0) - 1.96 * (ckpt_table['auc'].std(axis=0) / np.sqrt(len(ckpt_table))))
            auc_high_list.append(ckpt_table['auc'].mean(axis=0) + 1.96 * (ckpt_table['auc'].std(axis=0) / np.sqrt(len(ckpt_table))))
            
        plt.plot(ckpts, auc_mean_list, marker='o', markersize=5, label=plot_lables[methods.index(method)], color=colors[methods.index(method)])
        # add 95% confidence interval
        plt.fill_between(ckpts, auc_low_list, auc_high_list, alpha=0.2)

    plt.xlabel('Number training cycles [-]', fontproperties=verdana_font)
    plt.xticks(ckpts, fontproperties=verdana_font, fontsize=10)
    plt.yticks(fontproperties=verdana_font, fontsize=10)
    plt.ylim(0, 0.65)
    plt.ylabel('AUC@VR [-]', fontproperties=verdana_font)
    #set x_ticks rotation to 20
    # plt.xticks(rotation=20)
    plt.grid(axis='y')
    plt.legend(loc='lower right', fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f'ablation_{target}.png'))
    plt.show()

## real world results

In [ ]:
# compare planners: print AUC per method for experiment 2 (no plots)
import os
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

results_root = MATERIALS_DIR + '/NO_COLOR_RESULTS'
experiments = [2]
targets = ['PLANT', 'NODE', 'FRUIT']
planner_name = ['random_global', 'predefined_global', 'ssl_nbv', 'voxel_global_nbv', 'ssl_semantic_nbv_NOCOLOR']
planner_plot_name = ['Random','Pre-Defined','SSL-NBV','Voxel-NBV','SSL-Semantic-NBV']
colors = ['#1F77B4', '#2CA02C', '#FF7F0E' , '#9467BD', '#D62728' ]

def read_results_csv(path):
    if not os.path.exists(path):
        return pd.DataFrame()
    with open(path, 'r', newline='') as f:
        reader = csv.reader(f)
        rows = list(reader)
    if not rows:
        return pd.DataFrame()
    header = rows[0]
    valid_rows = [r for r in rows[1:] if len(r) == len(header)]
    return pd.DataFrame(valid_rows, columns=header)

def mean_ci95(series):
    s = pd.to_numeric(series, errors='coerce')
    s = s[~s.isna()]
    n = len(s)
    if n == 0:
        return float('nan'), float('nan')
    if n == 1:
        return float(s.iloc[0]), 0.0
    mean = float(s.mean())
    std = float(s.std())
    ci = 1.96 * (std / np.sqrt(n))
    return mean, ci

for experiment in experiments:
    print(f'======================experiment: {experiment}=========================')
    per_method_target_means = {method: [] for method in planner_name}
    for target in targets:
        results_dir = os.path.join(results_root, f'{experiment}_{target}')
        csv_path = os.path.join(results_dir, 'results.csv')
        df = read_results_csv(csv_path)
        print(f'-- target: {target} --')
        if df.empty:
            print('No data found')
            print()
            continue
        cov_cols = [c for c in df.columns if c.startswith('cov_')]
        cov_cols_sorted = sorted(cov_cols, key=lambda x: int(x.split('_')[1]))
        # Plot coverage ratio curves per planner
        plt.figure(figsize=(5.5, 3.5))
        for idx, (method, method_name) in enumerate(zip(planner_name, planner_plot_name)):
            df_planner = df[df['planner'] == method]
            if df_planner.empty or not cov_cols_sorted:
                continue
            coverage = df_planner[cov_cols_sorted].apply(pd.to_numeric, errors='coerce')
            coverage_mean = coverage.mean(axis=0)
            plt.plot(coverage_mean.values, label=method_name, color=colors[idx])
        plt.xlabel('Number of views [-]')
        plt.ylabel('Coverage ratio [-]')
        plt.grid()
        plt.legend(loc='lower right', fontsize=10)
        plt.tight_layout()
        plt.show()

        for method, method_name in zip(planner_name, planner_plot_name):
            auc_mean, auc_ci = mean_ci95(df[df['planner']==method]['auc'])
            if not np.isnan(auc_mean):
                per_method_target_means[method].append(auc_mean)
            print(f'{method_name} AUC: {auc_mean:.4f} ± {auc_ci:.4f} (95% CI)')
        # Print p-value for FRUIT: ssl-semantic-nbv vs voxel-nbv
        if target == 'FRUIT':
            ssl_semantic_auc = pd.to_numeric(df[df['planner']=='ssl_semantic_nbv_NOCOLOR']['auc'], errors='coerce')
            ssl_nbv_auc = pd.to_numeric(df[df['planner']=='ssl_nbv']['auc'], errors='coerce')
            ssl_semantic_auc = ssl_semantic_auc[~ssl_semantic_auc.isna()]
            ssl_nbv_auc = ssl_nbv_auc[~ssl_nbv_auc.isna()]
            if len(ssl_semantic_auc) > 0 and len(ssl_nbv_auc) > 0:
                t_stat, p_val = ttest_ind(ssl_semantic_auc, ssl_nbv_auc, equal_var=False)
                print(f"p-value for FRUIT: SSL-Semantic-NBV vs SSL-NBV: {p_val:.6f}")
            else:
                print("Not enough data to compute p-value for FRUIT: SSL-Semantic-NBV vs Voxel-NBV")
        print()
    print('Average AUC across targets (95% CI):')
    for method, method_name in zip(planner_name, planner_plot_name):
        target_means = per_method_target_means[method]
        if target_means:
            avg_series = pd.Series(target_means)
            avg_mean, avg_ci = mean_ci95(avg_series)
            print(f'{method_name} avg AUC: {avg_mean:.4f} ± {avg_ci:.4f} (95% CI)')
        else:
            print(f'{method_name} avg AUC: n/a')
    print()
